# Lesson 4：Tool、CLI、MCP 与 Skill —— 给 Agent 装上手脚

前几课的 Agent 只会「说」：Echo Agent 复述指令、CoT 一步步推理、RAG 检索资料。从本课开始，Agent 要能「**做**」：查天气、算数、执行命令、连接外部系统。

本课最重要的一条心智模型.

> **模型从不亲自执行任何操作。它只输出「我想调用哪个工具、参数是什么」的结构化意图（JSON），真正的执行永远发生在你的代码里，执行结果再作为「环境反馈」喂回给模型。**

围绕这条心智模型，业界演化出了给 Agent 配备能力的四种方式，也是本课的四个章节：

| 方式 | 一句话定义 | 核心载体 | 谁决定调用 | 类比 |
|---|---|---|---|---|
| **Tool**（工具 / 函数调用） | 模型可选调用的、带结构化参数的函数 | JSON Schema | 模型 | 岗位说明书 |
| **CLI**（命令行） | 通过进程参数 + stdin/stdout 暴露能力 | 文本约定 | 模型（经由 shell 工具） | 万能瑞士军刀 |
| **MCP**（Model Context Protocol） | 发现和调用外部能力的开放协议 | JSON-RPC（stdio / HTTP） | 模型（经 host 发现） | 工具世界的 USB-C |
| **Skill**（技能） | 按需加载的「说明书 + 脚本 + 资源」包 | SKILL.md（Markdown） | 模型决定何时读 | 一本操作手册 |

**学习目标**：完成本课后你将能够

1. ✅ 手写工具的 JSON Schema，并解释它和 Python 函数签名的分工
2. ✅ 从零实现一个完整的 Agent Loop（工具调用 → 执行 → 回传 → 最终回答）
3. ✅ 把命令行包装成带白名单的安全工具交给模型
4. ✅ 用 FastMCP 写一个 MCP Server，作为 Client 连接它，并桥接给 OpenAI 模型
5. ✅ 写一个符合规范的 Agent Skill，并解释渐进式披露（progressive disclosure）
6. ✅ 面对一个新需求，能判断该用 Tool / CLI / MCP / Skill 中的哪一个
7. ✅ 用 TongAgents SDK 对照实现每样能力：`@tool` / `ReactAgent` / `MCPClient` / `SkillTool`

> 内容参考 OpenAI、Anthropic 官方文档与 [claude-cookbooks](https://github.com/anthropics/claude-cookbooks) 开源仓库，链接见文末。

**本课采用双线结构**：每个主题先**手写白盒**（亲眼看协议长什么样），再用 **TongAgents SDK** 实现同一件事（各节尾部的「TongAgents 实现」小节）。黑盒只有和白盒对照着看，才知道框架替你做了什么、没做什么。

> TongAgents 是内部多智能体框架（Lesson 1/2 已用过它的 `Agent` 基类），从 BigAI Nexus 私有源安装，见 README；未安装时相关单元格会自动跳过。


## 0. 环境准备与自检

本目录由 [uv](https://docs.astral.sh/uv/) 管理（从零搭建的完整步骤见 [README.md](./README.md)）：

```bash
cd lesson4
uv sync                                       # 按 pyproject.toml + uv.lock 创建 .venv（版本已锁定，可完全复现）
cp .env.example .env                          # 复制后编辑 .env，填入你的 API Key
uv run jupyter lab lesson4_tools_mcp_cli_skill.ipynb
```

`.env` 只需要关心三个 OpenAI 环境变量（官方 OpenAI 或任何 OpenAI 兼容服务均可，见 README）：

| 变量 | 含义 | 默认值 |
|---|---|---|
| `OPENAI_API_KEY` | API Key（第 3 节起的部分示例需要） | — |
| `OPENAI_BASE_URL` | API 端点 | `https://api.openai.com/v1` |
| `OPENAI_MODEL` | 模型名 | `gpt-5.6-luna`（当前最便宜的工具调用模型） |

**没有 Key 也能学**：第 1、2、4.1、5.1–5.2、6.1 节完全本地运行；需要真实模型的单元格会自动跳过并提示，不影响后续单元格。


In [1]:
import importlib.metadata as im
import json, os, re, sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()                      # 把 .env 中的变量读入进程环境

import openai

API_KEY  = os.getenv("OPENAI_API_KEY", "")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
MODEL    = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
HAS_KEY  = bool(API_KEY) and API_KEY not in ("sk-your-api-key", "your-api-key")

print(f"Python    : {sys.version.split()[0]}（来自 uv 创建的 .venv）")
print(f"openai SDK: {openai.__version__}")
print(f"mcp SDK   : {im.version('mcp')}")
print(f"BASE_URL  : {BASE_URL}")
print(f"MODEL     : {MODEL}")
print(f"API Key   : {'✅ 已配置' if HAS_KEY else '❌ 未配置（需要真实模型的示例会自动跳过）'}")

# --- TongAgents SDK（内网 Nexus 安装，见 README；未安装则相关小节自动跳过）---
try:
    import tongagents
    HAS_TONGAGENTS = True
    print(f"tongagents {getattr(tongagents, '__version__', '?')} 就绪")
except ImportError:
    HAS_TONGAGENTS = False
    print("⏭️ 未安装 tongagents：2.5 / 3.4 / 5.5 / 6.3 / 6.4 节将自动跳过（安装见 README）")


Python    : 3.12.8（来自 uv 创建的 .venv）
openai SDK: 1.99.8
mcp SDK   : 1.26.0
BASE_URL  : https://open.bigmodel.cn/api/coding/paas/v4
MODEL     : glm-5.3
API Key   : ✅ 已配置
tongagents 2.7.20 就绪


## 1. 为什么 Agent 需要工具？

裸 LLM 有三个无法靠 prompt 解决的硬伤：

1. **知识是冻结的**——训练截止日期之后的世界它一无所知（今天的天气、最新的股价）；
2. **精确计算不可靠**——下一个 token 预测不擅长精确算术，`1234 × 5678` 很可能算错；
3. **只会说不会做**——它无法发邮件、查数据库、执行一条命令，对现实世界零影响。

工具调用范式把三个问题一次解决：模型负责**决策**（调用什么、传什么参数），代码负责**执行**（真正去算、去查、去做），结果回传给模型作为**环境反馈**。Anthropic 在经典文章《Building Effective Agents》里给 Agent 的定义正是这个循环：

> "Agents ... are typically just LLMs using tools based on environmental feedback in a loop."
> （Agent 通常只是在循环中、基于环境反馈使用工具的 LLM。）

把这句话翻译成代码，就是贯穿本课的 **Agent Loop**：

```text
用户提问
  → ① 模型推理，产出调用意图（tool_call）：
        {"name": "get_weather", "arguments": "{"city": "北京"}"}
  → ② 你的代码解析 JSON，真正执行函数，得到结果（tool_result）
  → ③ 把 tool_result 追加进对话历史，再次调用模型
  → ④ 模型基于结果继续推理：可能再次调用工具，也可能直接给出最终回答
（①②③④ 循环往复，直到模型认为任务完成）
```

同一个循环里，**流程控制权在谁手上**是区分两种架构的关键——也是《Building Effective Agents》最重要的判断：

> "Workflows are systems where LLMs and tools are orchestrated through **predefined code paths**."
> （工作流：LLM 和工具由**预先写死的代码路径**编排——流程在代码手里。）
>
> "Agents are systems where LLMs **dynamically direct their own processes** and tool usage."
> （Agent：LLM **动态主导自己的流程**和工具使用——流程在模型手里。）

Lesson 2 的 CoT / RAG 流水线更接近 workflow；本课要写的 Agent Loop 把「下一步做什么」交给模型。文章还有一句值得抄在团队 wiki 上的告诫：

> "Find the simplest solution possible, and only increase complexity when needed."
> （找最简单的可行方案，确有需要时再增加复杂度。）

工具接口值得像产品 UI 一样认真打磨——文章称之为 **ACI**（Agent-Computer Interface，人机接口 HCI 的对应物）：工具描述写得好不好，对效果的影响不亚于 prompt 本身。第 3.3 节会专门讲怎么写好工具。


## 2. Tool 基础：把 Python 函数变成模型能「看懂」的动作（无需 API Key）

一个工具 = **实现**（普通 Python 函数）+ **说明书**（JSON Schema）。函数签名是给 Python 解释器看的，Schema 是给模型看的——模型在每次请求时读到全部工具的说明书，据此决定要不要调用、怎么传参。

OpenAI 与 Anthropic 的工具定义几乎一样，只是字段名不同：

| | OpenAI `tools` | Anthropic `tools` |
|---|---|---|
| 工具名 / 描述 | `function.name` / `function.description` | `name` / `description` |
| 参数模式 | `function.parameters` | `input_schema` |
| 模型的调用意图 | `message.tool_calls[].function` | content 块 `tool_use` |
| 结果回传 | `role="tool"` 消息 + `tool_call_id` | `tool_result` 块 + `tool_use_id` |

> 本节结构：定义工具 → 分发执行 → 接模型。官方指南：[OpenAI Function Calling](https://developers.openai.com/api/docs/guides/function-calling)、[Anthropic Tool Use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)。

### 2.1 第一步：实现（就是普通函数）


In [2]:
# ---- 工具 1：查天气（模拟数据，聚焦机制本身）----
WEATHER_DB = {
    "北京": "晴，26°C，空气质量良",
    "上海": "多云，29°C，湿度 78%",
    "广州": "雷阵雨，31°C",
}

def get_weather(city: str) -> str:
    return WEATHER_DB.get(city, f"暂无 {city} 的天气数据")

# ---- 工具 2：计算器（先清洗再求值）----
def calculate(expression: str) -> str:
    """只保留数字和四则运算符，其余字符一律剔除，然后求值"""
    cleaned = re.sub(r"[^0-9+\-*/(). ]", "", expression)
    try:
        return str(eval(cleaned))   # ⚠️ 教学演示；生产环境请用 ast 或符号计算库
    except Exception:
        return "Error: 无效的表达式"

print(get_weather("北京"))
print(calculate("1234 * 5678"))


晴，26°C，空气质量良
7006652


### 2.2 第二步：说明书（手写 JSON Schema）+ 安全分发

注意 `description` 的写法——**不仅说这个工具是什么，还要说「什么时候该用它」**。模型全靠这段文字做决策，这是 ACI 打磨的第一现场。


In [3]:
tool_get_weather = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询某个城市当前的天气。当用户问到天气、温度、下雨、穿什么衣服时调用。",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "城市名，如：北京、上海"}
            },
            "required": ["city"],
        },
    },
}

tool_calculate = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "计算一个四则算术表达式。当需要精确数值计算时调用，不要心算。",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "算术表达式，如 '1234 * 5678'"}
            },
            "required": ["expression"],
        },
    },
}

TOOLS = [tool_get_weather, tool_calculate]

# 注册表：工具名 → 真正的实现。第 4 节往里加 CLI 工具，第 5 节加 MCP 工具
TOOL_REGISTRY = {"get_weather": get_weather, "calculate": calculate}

def dispatch_tool(name: str, arguments: dict) -> str:
    """把模型的调用意图变成真正的执行。

    关键设计：模型输出是「不可信输入」——
    1. 只认注册表里的白名单，不存在模型指定任意函数的通道；
    2. 任何异常都返回「可读的错误字符串」而不是抛异常崩溃，
       这样模型能看到错误并自我纠正（Anthropic 官方强烈推荐的做法）。
    """
    if name not in TOOL_REGISTRY:
        return f"Error: 未知工具 '{name}'，可用工具：{list(TOOL_REGISTRY)}"
    try:
        return str(TOOL_REGISTRY[name](**arguments))
    except TypeError as e:
        return f"Error: 参数不匹配，请检查参数名和数量（{e}）"
    except Exception as e:
        return f"Error: {e}"

print(json.dumps(tool_get_weather, ensure_ascii=False, indent=2))


{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "查询某个城市当前的天气。当用户问到天气、温度、下雨、穿什么衣服时调用。",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "城市名，如：北京、上海"
        }
      },
      "required": [
        "city"
      ]
    }
  }
}


### 2.3 模型「说的话」长什么样？（手动模拟大模型返回的tool call list）

先不调 API。下面手动构造一次模型返回的 `tool_calls`——**形状和真实 API 返回的一模一样**，先用它打通「意图 → 执行 → 回传」整条链路，理解机制后再接真模型。

两个细节：

- `arguments` 是 **JSON 字符串**，不是 dict，必须 `json.loads` 一下；
- 每个调用有唯一的 `id`，回传结果时靠它配对——一轮可以有**多个** `tool_call`（并行工具调用）。


In [4]:
simulated_tool_calls = [
    {"id": "call_abc123", "type": "function",
     "function": {"name": "get_weather", "arguments": json.dumps({"city": "北京"}, ensure_ascii=False)}},
    {"id": "call_def456", "type": "function",
     "function": {"name": "calculate", "arguments": json.dumps({"expression": "1234*5678"})}},
]

for tc in simulated_tool_calls:            # 一条消息里多个 tool_call = 并行工具调用
    name, raw = tc["function"]["name"], tc["function"]["arguments"]
    result = dispatch_tool(name, json.loads(raw))
    print(f"模型想调用: {name}({raw})")
    print(f"  执行结果 : {result}")
    print(f"  回传方式 : messages.append({{'role': 'tool', 'tool_call_id': '{tc['id']}', 'content': ...}})\n")


模型想调用: get_weather({"city": "北京"})
  执行结果 : 晴，26°C，空气质量良
  回传方式 : messages.append({'role': 'tool', 'tool_call_id': 'call_abc123', 'content': ...})

模型想调用: calculate({"expression": "1234*5678"})
  执行结果 : 7006652
  回传方式 : messages.append({'role': 'tool', 'tool_call_id': 'call_def456', 'content': ...})



### 2.4 安全边界：模型输出是不可信输入

三层防线，缺一不可（对应 [Writing tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents) 中的建议）：

1. **白名单分发**：模型只能点名注册表里的工具名，任何其他的都是非法的；
2. **参数清洗**：进 `eval` 前剥离一切非算术字符；
3. **错误返回给模型**：返回「可操作的错误信息」（附上正确格式示例更好），模型会据此重试，而不是让你的程序崩溃。


In [5]:
# 三连攻击演示：都没能突破防线
print(calculate("__import__('os').system('rm -rf /')"))     # ① 危险字符被白名单剔除，变成无害（但无效）的算式
print(dispatch_tool("os.system", {"command": "rm -rf /"}))   # ② 不在注册表 → 拒绝
print(dispatch_tool("get_weather", {}))                      # ③ 缺参数 → 可操作的错误信息（模型可据此补参重试）


Error: 无效的表达式
Error: 未知工具 'os.system'，可用工具：['get_weather', 'calculate']
Error: 参数不匹配，请检查参数名和数量（get_weather() missing 1 required positional argument: 'city'）


### 2.5 TongAgents 实现：`@tool` 装饰器 —— 自动生成你刚才手写的一切

前四节我们手动完成了三件事：**写函数**（2.1）、**写 Schema**（2.2）、**写分发与安全**（2.3/2.4）。TongAgents SDK 的 `@tool` 装饰器把前两件合并成一步：框架解析函数签名和 docstring，自动生成 JSON Schema，并注册进全局 `ToolManager`。

它生成的字段和你在 2.2 手写的**完全同构**（name / description / parameters）——因为你手写的正是行业标准本身，`@tool` 只是把这份重复劳动自动化了。2.4 的安全分发在 SDK 里对应 `ToolHook`（`check_auth` 权限检查、`pre_hook` 参数注入、`filter_schema` 对模型隐藏参数），练习 8 会让你实现一个。

In [6]:
# TongAgents 的 @tool：签名 + docstring → JSON Schema（对照 2.2 的手写版）
if HAS_TONGAGENTS:
    from tongagents.tools.tool_manager import tool

    # 注意：这里同名覆盖了 2.1 的手写 get_weather —— 无妨，
    # 2.3 的 TOOL_REGISTRY 注册时已捕获旧函数引用，手写链路不受影响。
    @tool(retry_times=2)                     # retry_times：失败时允许模型重试的次数
    def get_weather(city: str):
        """查询指定城市的实时天气

        Args:
            city: 城市名，如"北京"
        """
        return WEATHER_DB.get(city, f"{city}：暂无数据")

    @tool()
    def calculate(expression: str):
        """计算一个算术表达式，如 "1234*5678"

        Args:
            expression: 算术表达式字符串
        """
        return str(eval(re.sub(r"[^0-9+\-*/(). ]", "", expression)))

    print("=== @tool 自动生成的 schema（对照 2.2 手写的 tool_get_weather）===")
    print(json.dumps(get_weather.parameters.to_json_schema(), ensure_ascii=False, indent=2))

    print("\n装饰后的工具仍可直接当函数调用:", get_weather(city="北京"))
    print("同时已自动注册进全局 ToolManager（ReactAgent 按名字就能找到它）")
else:
    print("⏭️ 未安装 tongagents，跳过本节（README 有安装说明）。")


=== @tool 自动生成的 schema（对照 2.2 手写的 tool_get_weather）===
{
  "properties": {
    "city": {
      "title": "City",
      "type": "string",
      "description": "城市名，如\"北京\""
    }
  },
  "required": [
    "city"
  ],
  "title": "get_weather",
  "type": "object",
  "description": "查询指定城市的实时天气"
}

装饰后的工具仍可直接当函数调用: 晴，26°C，空气质量良
同时已自动注册进全局 ToolManager（ReactAgent 按名字就能找到它）


## 3. 接入真实模型：Tool Calling 与 Agent Loop（需要 API Key）

Chat Completions 的 tool calling 全流程，其实就是把第 2 节的模拟变成真的：

1. 请求带上 `tools`（Schema 列表）和 `tool_choice="auto"`（模型自己决定是否调用）；
2. 模型若决定用工具 → 响应里 `message.tool_calls` 带一个或多个调用意图，`finish_reason == "tool_calls"`；
3. 你逐个执行，把结果以 `role="tool"` + `tool_call_id` 追加进 messages；
4. 再次请求。模型可能继续要工具，也可能直接回答（`finish_reason == "stop"`）。


In [7]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def chat_once(question: str, tools: list, tool_choice: str = "auto") -> None:
    """只观察模型的「调用意图」，暂不执行——亲眼确认：模型只出意图，不出结果。"""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice=tool_choice,   # auto=模型自己决定 / "required"=必须调工具 / 指定具体工具见文末练习
    )
    msg = resp.choices[0].message
    print(f"finish_reason = {resp.choices[0].finish_reason}")
    for tc in msg.tool_calls or []:
        print(f"  模型想调用: {tc.function.name}({tc.function.arguments})  [id={tc.id}]")
    if not msg.tool_calls:
        print(f"  模型直接回答: {msg.content}")

if HAS_KEY:
    chat_once("北京今天天气怎么样？顺便帮我算一下 1234*5678", TOOLS)   # 观察点：一轮里出现几个 tool_call？
else:
    print("⏭️ 未配置 OPENAI_API_KEY，已跳过。填好 .env 后重跑本单元格即可。")


finish_reason = tool_calls
  模型想调用: get_weather({"city":"北京"})  [id=call_-7293277024670447933]
  模型想调用: calculate({"expression":"1234*5678"})  [id=call_-7293277024670447932]


上面这个问题应该产生**两个** `tool_call`（查天气 + 算数）——这就是**并行工具调用**（parallel tool calling），两家 API 都原生支持：模型一次推理同时发起多个互不依赖的调用，Agent 少跑一轮循环。

### 3.2 完整 Agent Loop：把定义变成 15 行代码


In [8]:
def agent_loop(question: str, tools=None, registry=None, system=None,
               max_turns: int = 6, verbose: bool = True):
    """《Building Effective Agents》定义的最小实现：
    'LLMs using tools based on environmental feedback in a loop'"""
    tools = TOOLS if tools is None else tools
    registry = TOOL_REGISTRY if registry is None else registry
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, tool_choice="auto")
        msg = resp.choices[0].message

        if not msg.tool_calls:                       # 模型不再要工具 → 最终回答
            if verbose:
                print(f"[turn {turn}] 🏁 最终回答\n{msg.content}")
            return msg.content

        messages.append(msg)                         # 回传 assistant 消息（含 tool_calls）
        for tc in msg.tool_calls:                    # 一轮可能有多个调用（并行）
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            if name in registry:
                result = str(registry[name](**args))
            else:
                result = f"Error: 未知工具 '{name}'"
            if verbose:
                print(f"[turn {turn}] 🔧 {name}({args})\n        → {result[:120]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "（达到最大轮数，停止）"

if HAS_KEY:
    agent_loop("上海和北京哪个城市更热？请先查两地天气再比较。")
    # 观察点：模型先在一轮里并行查两个城市，再基于结果给出比较 —— 两轮循环完成任务
else:
    print("⏭️ 未配置 OPENAI_API_KEY，已跳过。")


[turn 1] 🔧 get_weather({'city': '上海'})
        → 多云，29°C，湿度 78%
[turn 1] 🔧 get_weather({'city': '北京'})
        → 晴，26°C，空气质量良
[turn 2] 🏁 最终回答
查询结果如下：

**天气对比：**

| 城市 | 天气 | 温度 | 其他 |
|------|------|------|------|
| 上海 | 多云 | 29°C | 湿度 78% |
| 北京 | 晴 | 26°C | 空气质量良 |

**结论：上海更热。** 上海当前气温为 29°C，比北京的 26°C 高出 3°C。不过上海湿度较高（78%），体感可能会更加闷热；北京虽然是晴天但温度略低，且空气质量良好，相对更舒适一些。


### 3.3 怎么把工具写好？官方最佳实践精选

来自 Anthropic 工程博客 [Writing tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents)（该文用「让 Agent 优化给 Agent 用的工具」做评估驱动迭代，结论都经过实测验证）：

1. **像给新同事介绍工具那样写 description**——把隐含约定写显式：参数格式、术语定义、资源之间的关系。文中案例：仅精修工具描述，模型在 SWE-bench Verified 上达到当时 SOTA。
2. **参数名要无歧义**：用 `user_id` 而不是 `user`；能用 `enum` 约束就不要靠模型自觉。
3. **合并工具优于堆砌工具**：一个 `schedule_event` 好过 `list_users` + `list_events` + `create_event` 三件套；`search_logs` 好过 `read_logs`。工具太多或互相重叠会分散模型注意力。
4. **控制工具输出的 token**：分页、截断、提供 `response_format: "concise" | "detailed"` 这样的枚举让模型自己选粒度（官方实测可省 2/3 token）；Claude Code 默认把单个工具输出截断在 25,000 token。
5. **错误信息要可操作**：附上正确格式的示例。评估里大量「无效参数」报错，通常说明 description 写得不够清楚，而不是模型太笨。
6. **用 evals 驱动迭代**：构造需要多次工具调用的真实任务，跑分后读完整 transcript（不要只看准确率）——冗余调用多就去调分页参数，参数报错多就去改描述。

OpenAI 官方指南的补充要点：

- **实习生测试（intern test）**：工具描述要写到「一个聪明实习生不用问你就能正确使用」的程度——和第 1 条异曲同工；
- **别让模型填你的代码已知的参数**：会话里已有 `user_id` 就自己注入，不要让模型编；
- **工具定义会计入 input tokens 并计费**（随请求注入上下文）——初始工具数建议少于 20 个，太多时用工具检索/命名空间（`tool_search`）按需加载；
- `parallel_tool_calls` 参数可以关闭并行（需要顺序保证的场景）；`strict: true`（Structured Outputs）可约束参数 100% 符合 Schema。


### 3.4 TongAgents 实现：`ReactAgent` —— 你手写的循环的工业化版本

你在 3.2 手写的 `agent_loop`（约 25 行），在 TongAgents 里叫 `ReactAgent`。它做的事一模一样——模型出意图 → 框架执行工具 → 结果回传 → 循环直到最终回答——只是多带三样生产级配套：**记忆**（`SimpleMemory` 自动记录全部消息，不用手动 `append`）、**反思重试**（工具抛 `RetryableToolCallException` 时让模型重新生成）、**流式/异步接口**（`stream` / `astep`）。

两个细节：

1. `ModelConfig()` 的三个参数**留空即自动读取** `OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL` 环境变量——正是 §0 里 `load_dotenv()` 加载的 `.env`；
2. 默认 provider 的底层实现就是 `OpenAI(base_url=...)`——**和你 3.1 手写用的是同一个客户端**。框架没有魔法，只是替你写了循环。

In [9]:
# ReactAgent：3.2 手写 agent_loop 的框架版（需要 Key + tongagents）
if not HAS_TONGAGENTS:
    print("⏭️ 未安装 tongagents，跳过本节。")
elif not HAS_KEY:
    print("⏭️ 未配置 OPENAI_API_KEY，跳过本节。")
else:
    from tongagents.agents.llm.base import ModelConfig
    from tongagents.agents.llm_agent.react_agent import LLMRunContext, ReactAgent, ReactAgentSetting

    mc = ModelConfig()  # 三个参数留空 → 自动读 OPENAI_API_KEY / OPENAI_BASE_URL / OPENAI_MODEL
    print(f"ModelConfig 自动吃到 .env：model={mc.model_name} | url={mc.url}")

    ctx = LLMRunContext()  # 持有 SimpleMemory：框架版的 messages 列表
    agent = ReactAgent(
        dep_context=ctx,
        agent_settings=ReactAgentSetting(llm_config=mc),
        tools=[get_weather, calculate],  # 复用 2.5 的两个 @tool 工具
    )
    result = agent.step("北京今天天气怎么样？顺便帮我算一下 1234*5678")
    print("".join(m.content for m in result if getattr(m, "content", None)))

    def dump_memory(ctx, width=70):
        """把框架自动记录的消息轨迹剥出来看（内部结构较深，逐层拆包）"""
        for rec in ctx.memory.get_messages():
            for ch in getattr(rec, "chunks", []):
                m = getattr(ch, "content", None)
                t = type(m).__name__
                if t == "UserPromptMessage":
                    print(f"  [user]      {str(m.content)[:width]}")
                elif t == "ModelOutputMessage":
                    for tc in m.tool_call_messages or []:
                        print(f"  [assistant] 发起调用 {tc.tool_name}({tc.arguments})")
                    if m.model_text_message and m.model_text_message.content:
                        print(f"  [assistant] {str(m.model_text_message.content)[:width]}")
                elif t == "ToolResultMessage":
                    print(f"  [tool]      {m.name} → {str(m.content)[:width - 15]}")
                else:
                    print(f"  [{t}] {str(m)[:width]}")

    print("\n=== SimpleMemory 自动记录的完整轨迹（对照 3.2 手动 append 的 messages）===")
    dump_memory(ctx)


/tmp/ipykernel_2227271/2171227618.py:8: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  from tongagents.agents.llm_agent.react_agent import LLMRunContext, ReactAgent, ReactAgentSetting


ModelConfig 自动吃到 .env：model=glm-5.3 | url=https://open.bigmodel.cn/api/coding/paas/v4
北京今天天气不错！☀️

**北京天气：**
- 天气状况：晴
- 气温：26°C
- 空气质量：良

**计算结果：**
1234 × 5678 = **7006652**

今天天气很适合外出活动，祝你一天愉快！

=== SimpleMemory 自动记录的完整轨迹（对照 3.2 手动 append 的 messages）===
  [user]      北京今天天气怎么样？顺便帮我算一下 1234*5678
  [assistant] 发起调用 get_weather({"city":"北京"})
  [assistant] 发起调用 calculate({"expression":"1234*5678"})
  [tool]      get_weather → 晴，26°C，空气质量良
  [tool]      calculate → 7006652
  [assistant] 北京今天天气不错！☀️

**北京天气：**
- 天气状况：晴
- 气温：26°C
- 空气质量：良

**计算结果：**
1234 × 5


## 4. CLI：把命令行变成 Agent 的工具

CLI 不是为大模型设计的接口，却是**最通用**的能力暴露方式：Claude Code、Codex 这类编程 Agent 的核心，本质就是「模型 + 一个 shell 工具」——任何能在终端里用的能力，Agent 都能用，一个 schema 都不用写。

| | Function Tool | CLI |
|---|---|---|
| 参数 | JSON Schema 强约束 | 自由文本，靠 `--help` 和惯例约定 |
| 发现方式 | 显式注册进 `tools` | 模型靠训练语料里学到的命令知识 |
| 组合能力 | 一次调用一个工具 | 管道 `\|`、重定向、脚本组合 |
| 安全控制 | 按工具粒度白名单 | **必须**按命令粒度白名单（一条 `rm -rf` 就是任意代码执行） |

代价是风险敞口完全不同：函数工具的参数只在你的函数里流动，而 shell 命令本身就是代码。所以演示版 `run_command` 内置三层防护：**命令白名单 + 超时 + 输出截断**（真实工具如 Claude Code 还要加沙箱和逐条审批，见 4.3）。

### 4.1 写一个带白名单的 shell 工具（无需 Key）


In [ ]:
import shlex
import subprocess

ALLOWED_COMMANDS = {"ls", "cat", "head", "tail", "wc", "du", "df", "grep", "find", "python3"}
MAX_OUTPUT_CHARS = 2000        # 工具输出会进入模型上下文 → 必须截断（3.3 节第 4 条实践）

def run_command(command: str) -> str:
    """受控执行一条命令：白名单 + 超时 + 输出截断。"""
    parts = shlex.split(command)
    if not parts or parts[0] not in ALLOWED_COMMANDS:
        head = parts[0] if parts else ""
        return (f"Error: 命令 '{head}' 不在白名单内。"
                f"可用命令：{sorted(ALLOWED_COMMANDS)}")
    try:
        proc = subprocess.run(parts, capture_output=True, text=True, timeout=15)
        output = (proc.stdout + proc.stderr)[:MAX_OUTPUT_CHARS]
        return f"[exit={proc.returncode}]\n{output}"
    except subprocess.TimeoutExpired:
        return "Error: 命令执行超时（>15s），请换更小的范围重试"

print(run_command("ls"))
print(run_command("python3 -c 'print(6*7)'"))
print(run_command("rm -rf /"))          # 被白名单拒绝


### 4.2 注册给模型：Agent 自己用命令行干活

给 `run_command` 写一份 Schema、加进注册表——注意和第 2 节一样是**普通注册**，Agent Loop 一行都不用改。这就是良好抽象的回报：加能力 = 注册一条。


In [ ]:
TOOL_REGISTRY["run_command"] = run_command
TOOLS.append({
    "type": "function",
    "function": {
        "name": "run_command",
        "description": (
            "在工作目录执行一条 shell 命令并返回输出。只允许这些命令："
            "ls/cat/head/tail/wc/du/df/grep/find/python3。"
            "需要查看文件、统计大小、搜索文本时使用。"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "完整命令行，如 'ls -lh' 或 'du -h uv.lock'"}
            },
            "required": ["command"],
        },
    },
})

if HAS_KEY:
    agent_loop("当前目录下都有哪些文件？其中最大的文件是哪个、有多大？")
    # 观察点：模型可能先 ls 看全貌，再用 du/ls -lh 查大小 —— 它在用多轮循环逼近答案
else:
    print("⏭️ 未配置 OPENAI_API_KEY，已跳过（4.1 无需 Key 已可运行）。")


### 4.3 现实中的 CLI Agent：一条「能力接入频谱」

把本课的内容放到一根数轴上看，2026 年的官方叙事其实是「能力接入频谱」：

| 方式 | 谁写代码 | 耗时量级 | 例子 |
|---|---|---|---|
| Function tool | 你写被调用的代码 | 毫秒级 | `get_weather` |
| Code interpreter（API 内置沙箱） | 模型写 Python | 秒级 | OpenAI `code_interpreter` 工具 / Claude 的代码执行 |
| CLI Agent | 模型全权操作终端与文件系统 | 分钟级 | Claude Code、OpenAI Codex CLI |

生产环境的 shell 工具远比 4.1 的演示版谨慎，以 Claude Code 为例：默认沙箱限制文件系统/网络写操作、高危命令逐条向用户请求批准、输出超限自动截断。设计你自己的 CLI 工具时，**白名单是最小要求，不是终点**。


### 4.4 TongAgents CLI：把整个 Agent 装进命令行

我们的 `run_command` 是「给模型一把螺丝刀」；TongAgents CLI 反过来：**把 Agent 本身做成命令行工具**，你在终端里直接使唤它：

```bash
tongagents run my_workflow.yaml     # 执行一个 workflow
tongagents interactive              # 进入交互式对话（Agent 的 REPL）
```

（`tongagents-cli` 是独立包，Lesson 1 的 requirements 已安装。）CLI 不再是 Agent 的工具，CLI **就是** Agent 的外壳——Claude Code、Codex CLI 与它是同类生物。

## 5. MCP（Model Context Protocol）：工具接入的开放协议

第 2 节的方式有个隐藏成本：工具 Schema 和实现要**手动注册**进每一个 Agent 应用。如果有 M 个应用（Claude Code、Cursor、你的 notebook……）要用 N 个能力（天气、数据库、GitHub……），就要写 M×N 份胶水代码。

MCP（Anthropic 2024 年 11 月开源，现已是社区共同演进的开放标准）把接口标准化成一份协议，M×N 变成 **M+N**：能力方写一次 Server，应用方实现一次 Client，即插即用——官方类比是「AI 应用的 USB-C 接口」。

**三个角色**（[官方架构文档](https://modelcontextprotocol.io/docs/concepts/architecture)）：

- **Host**：AI 应用（Claude Code、Cursor，或本课的 notebook），协调一个或多个 Client；
- **Client**：host 内维护与某个 server 专属连接的组件；
- **Server**：提供能力的程序（本地子进程或远程服务）。

**Server 能提供的三类原语**：`tools`（模型可调用的动作）、`resources`（上下文数据，如文件内容、数据库 schema）、`prompts`（可复用的交互模板）。

**两种传输**：`stdio`（拉起本地子进程，标准输入输出上跑 JSON-RPC，本课演示）和 `Streamable HTTP`（远程服务，HTTP POST + SSE，可服务大量客户端）。

生态现状：Claude Code 用 `.mcp.json` 配置 server；OpenAI Agents SDK 内置 `MCPServerStdio` 支持；GitHub、Sentry、Cloudflare 等都提供官方 server。

### 5.1 十行写一个 MCP Server（无需 Key）

对比第 2 节手写 Schema：FastMCP 从**类型注解 + docstring** 自动生成 JSON Schema——说明书和实现不会再不同步。


In [12]:
%%writefile mcp_weather_server.py
"""最小 MCP Server：FastMCP 把类型注解和 docstring 自动变成 JSON Schema。

运行方式：不由用户手动启动，而是作为子进程被 MCP client 拉起（stdio 传输）。
"""
import logging

from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.ERROR)        # 教学时保持输出干净：关掉 INFO 级请求日志

mcp = FastMCP("weather-station")


@mcp.tool()
def get_weather(city: str) -> str:
    """查询某个城市当前的天气。"""
    db = {"北京": "晴，26°C", "上海": "多云，29°C", "广州": "雷阵雨，31°C"}
    return db.get(city, f"暂无 {city} 的天气数据")


@mcp.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """查询城市未来几天的天气预报。days 默认 3 天，最多 7 天。"""
    days = max(1, min(days, 7))
    return f"{city} 未来 {days} 天：晴转多云，气温 24~30°C（模拟数据）"


if __name__ == "__main__":
    mcp.run()        # 默认 stdio 传输


Overwriting mcp_weather_server.py


### 5.2 作为 Client 连接：发现与调用（无需 Key）

三个协议动作——`initialize`（握手：交换身份、协议版本、能力清单）、`tools/list`（发现工具）、`tools/call`（调用工具）。底层都是 JSON-RPC 2.0 消息，SDK 帮我们封装好了。

两个实战细节：

- 启动子进程必须用 `sys.executable`（当前 venv 的 Python），写成 `"python"` 在很多环境会 `FileNotFoundError`；
- **连接的生命周期用 `async with` 管理、在一个单元格内完整闭合**：Jupyter 的每个单元格是独立的异步任务，而 `async with` 的上下文（内部的 cancel scope）绑定创建它的任务——跨单元格「先开连接、后关连接」会直接报 `RuntimeError`。正式应用里通常用一个长驻后台任务持有连接，本课保持简单。

连接与模型完全无关，**这一节不需要 API Key**。


In [13]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,            # ⚠️ 用当前 venv 的 Python 拉起 server 子进程
    args=["mcp_weather_server.py"],
)

async with stdio_client(SERVER_PARAMS) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()                     # ① 握手
        tool_list = await session.list_tools()         # ② 发现：tools/list
        print("server 提供的工具：")
        for t in tool_list.tools:
            print(f"  🔧 {t.name}{list(t.inputSchema.get('properties', {}))} —— {t.description}")
        print("\nget_forecast 的 Schema（自动从类型注解生成）：")
        print(json.dumps(tool_list.tools[1].inputSchema, ensure_ascii=False, indent=2))

        r1 = await session.call_tool("get_weather", {"city": "上海"})   # ③ 调用：tools/call
        r2 = await session.call_tool("get_forecast", {"city": "北京", "days": 5})
        print(f"\n{r1.content[0].text}\n{r2.content[0].text}")
        print(f"isError: {r1.isError}, {r2.isError}")
        # 返回被统一包装成 content 列表（可含文本/图片/资源）——协议「与模型无关」的抽象层
# 离开 async with：会话与 server 子进程自动清理


server 提供的工具：
  🔧 get_weather['city'] —— 查询某个城市当前的天气。
  🔧 get_forecast['city', 'days'] —— 查询城市未来几天的天气预报。days 默认 3 天，最多 7 天。

get_forecast 的 Schema（自动从类型注解生成）：
{
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    },
    "days": {
      "default": 3,
      "title": "Days",
      "type": "integer"
    }
  },
  "required": [
    "city"
  ],
  "title": "get_forecastArguments",
  "type": "object"
}

多云，29°C
北京 未来 5 天：晴转多云，气温 24~30°C（模拟数据）
isError: False, False


### 5.3 桥接：让 OpenAI 模型用上 MCP 工具（需要 Key）

MCP 只负责「发现 + 调用」，完全不关心上层是什么模型。桥接三步：

1. `list_tools()` 的结果转成 OpenAI 的 `tools` 格式（只是字段换个名字）；
2. 模型返回 `tool_call` → 转成 `session.call_tool()`；
3. 结果塞回 `role="tool"` 消息。

OpenAI Agents SDK 的 `MCPServerStdio.get_tools()` 内部做的就是这件事——我们自己写一遍，以后用任何框架都心里有数。


In [14]:
def mcp_tools_to_openai(tool_list) -> list:
    """MCP 工具 → OpenAI tools 格式（字段换个名字而已）"""
    return [{"type": "function",
             "function": {"name": t.name, "description": t.description or "",
                          "parameters": t.inputSchema}}
            for t in tool_list.tools]

async def mcp_agent_loop(question: str, session, max_turns: int = 6) -> str:
    """Agent Loop 的 MCP 版：唯一区别是执行从 registry 查表变成 session.call_tool"""
    tools = mcp_tools_to_openai(await session.list_tools())
    messages = [{"role": "user", "content": question}]
    for turn in range(1, max_turns + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            print(f"[turn {turn}] 🏁 {msg.content}")
            return msg.content
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            r = await session.call_tool(tc.function.name, args)      # 关键差异行
            text = "\n".join(getattr(c, "text", "") for c in r.content)
            print(f"[turn {turn}] 🔧 {tc.function.name}({args})\n        → {text[:100]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": text})
    return "（达到最大轮数）"

if HAS_KEY:
    async with stdio_client(SERVER_PARAMS) as (read, write):        # 再开一条连接
        async with ClientSession(read, write) as session:
            await session.initialize()
            await mcp_agent_loop("广州未来一周天气如何？这个周末适合户外露营吗？", session)
else:
    print("⏭️ 未配置 OPENAI_API_KEY，已跳过（5.1/5.2 无需 Key 已可运行）。")


[turn 1] 🔧 get_forecast({'city': '广州', 'days': 7})
        → 广州 未来 7 天：晴转多云，气温 24~30°C（模拟数据）
[turn 2] 🏁 根据广州未来 7 天的天气预报，情况如下：

## 📅 未来一周天气概况
- **天气**：晴转多云
- **气温**：24~30°C

## 🏕️ 周末露营建议
这个周末**总体适合户外露营**，理由如下：

✅ **优势：**
- 晴到多云的天气，没有降雨迹象，不用担心淋雨
- 24~30°C 的温度比较舒适，白天温暖，夜间略有凉意（露营睡觉体感更佳）
- 适合白天徒步、烧烤、搭帐篷等活动

⚠️ **温馨提示：**
1. **防晒**：白天最高 30°C，紫外线可能较强，建议带好防晒霜、遮阳帽
2. **补水**：气温较高时户外活动要多喝水
3. **防蚊虫**：广州气候湿热，露营时记得带驱蚊液
4. **装备**：夜间温度可能降到 24°C 左右，带一件薄外套会更舒适
5. **帐篷选择**：白天较热，建议选通风性好的帐篷

祝您周末露营愉快！🌿⛺


### 5.4 现实中怎么接入 MCP

**Claude Code**（项目根目录 `.mcp.json`，或 `claude mcp add` 命令）：

```json
{
  "mcpServers": {
    "weather": {"command": "python", "args": ["mcp_weather_server.py"]}
  }
}
```

**OpenAI Agents SDK**（桥接全自动，相当于 5.3 我们手写部分的官方版）：

```python
from agents import Agent, Runner
from agents.mcp import MCPServerStdio

async with MCPServerStdio(params={"command": sys.executable, "args": ["mcp_weather_server.py"]}) as server:
    agent = Agent(name="helper", instructions="你是天气助手",
                  mcp_servers=[server])          # 工具发现和桥接全自动
    result = await Runner.run(agent, "上海会下雨吗？")
```

**Responses API 直接挂远程 MCP**（完全托管：工具发现、调用、结果回传都在 OpenAI 服务端完成）：

```python
resp = client.responses.create(
    model=MODEL,
    tools=[{"type": "mcp", "server_label": "weather",
            "server_url": "https://your-server.example/mcp",
            "require_approval": "never"}],
    input="上海会下雨吗？")
```

同一件事的三个层次：**手动桥接（我们刚写的）→ SDK 自动化（Agents SDK）→ 服务端托管（Responses API）**。抽象程度越高、可控性越低——教学和生产都遵循同一权衡。

**什么时候用 Tool，什么时候上 MCP？** 能力只在一个应用内部用 → 直接注册 Tool 最简单；能力需要被多个应用复用、独立部署、跨团队共享（或要用现成的第三方 server）→ 值得做成 MCP Server。注意 MCP 不是「更好的 Tool」——多一层进程和协议，本地单一场景反而更复杂。


### 5.5 TongAgents 接入 MCP：客户端 + 桥接，框架都替你写好了

回顾 5.2/5.3：我们手写了**连接生命周期管理**（三层 `async with` 嵌套）和 **MCP→OpenAI 工具桥接**（`mcp_tools_to_openai` 约 20 行）。TongAgents 把它们封装成 `MCPClient` 和 `MCPToolManager`，stdio / SSE / HTTP 三种传输只用换一个参数。


In [15]:
# TongAgents 接入 MCP：5.2 手写客户端 + 5.3 手写桥接 的框架版
if not HAS_TONGAGENTS:
    print("⏭️ 未安装 tongagents，跳过本节。")
elif not HAS_KEY:
    print("⏭️ 未配置 OPENAI_API_KEY，跳过本节。")
else:
    from concurrent.futures import ThreadPoolExecutor

    from tongagents.agents.llm.base import ModelConfig
    from tongagents.agents.llm_agent.react_agent import LLMRunContext, ReactAgent, ReactAgentSetting
    from tongagents.tools.mcp_client import MCPClient
    from tongagents.tools.mcp_tool_manager import MCPToolManager

    def run_mcp_demo():
        # ① 连接 stdio server。
        #  必须用 `uv run jupyter lab` 启动（PATH 里有 venv 的 python）。
        client = MCPClient("mcp_weather_server.py", timeout=15)
        print("发现的工具:", [t.name for t in client.list_tools()])

        # ② MCP 工具 → TongAgents Tool。
        mcp_tools = [T() for T in MCPToolManager.initialize_from_mcp(client)]
        # 下面的 WARNING 来自全局 ToolManager：2.5 注册的本地 get_weather
        # 被 MCP 同名工具覆盖 —— 全局注册表按名字索引，同名即冲突，留意即可。
        print("桥接完成:", [t.name for t in mcp_tools])

        # ③ 交给 ReactAgent —— 5.2+5.3 手写约 40 行的事，这里 3 行
        agent = ReactAgent(
            dep_context=LLMRunContext(),
            agent_settings=ReactAgentSetting(llm_config=ModelConfig()),
            tools=mcp_tools,
        )
        result = agent.step("上海今天天气怎么样？未来两天的预报呢？")  # 观察点：两个工具都会被用上吗？
        print("".join(m.content for m in result if getattr(m, "content", None)))

    # Jupyter 内核自身运行在事件循环里，MCPClient 的同步封装会返回
    # 解法：整段放进没有事件循环的独立线程（对照 5.2 的教训）。
    with ThreadPoolExecutor(max_workers=1) as ex:
        ex.submit(run_mcp_demo).result()


发现的工具: ['get_weather', 'get_forecast']


[WARNING] [2026-09-02 14:47:17,525] [tongagents] Tool `get_weather` already exists! Overwriting.
[WARNING] [2026-09-02 14:47:17,526] [tongagents] Tool `get_forecast` already exists! Overwriting.


桥接完成: ['get_weather', 'get_forecast']
我来帮您查询上海今天的天气和未来两天的预报。为您查询到上海的天气情况：

**今天：** 多云，气温 29°C ☁️

**未来两天预报：** 晴转多云，气温在 24~30°C 之间 🌤️

总体来说天气不错，温度适宜，未来两天以晴到多云为主，早晚温差稍大（最低24°C），建议适当增减衣物，适合外出活动！


## 6. Skill：给 Agent 的「操作手册」

前面三种方式解决「模型**能做**什么」（capability），Skill 解决「模型**知道怎么做**什么」（procedural knowledge）：公司周报的格式规范、课程总结的标准结构、某类分析的固定流程。这些知识塞 prompt 太浪费（每个请求都带着），写进工具又不合适——它们不是「动作」。

Agent Skill 的答案：**一个文件夹 = SKILL.md 说明书 + 可选的 scripts/、references/ 资源**，由模型**按需加载**。2025 年 12 月起它已成为开放标准（[agentskills.io](https://agentskills.io)），Claude Code（`.claude/skills/`）、Codex（`.agents/skills/`）、claude.ai 的内置文档技能都在用同一格式。

**渐进式披露（progressive disclosure）**——Skill 能打包几乎无限知识而不撑爆上下文的关键，官方分三级：

| 级别 | 加载内容 | 时机 | 上下文成本 |
|---|---|---|---|
| **1** | frontmatter 的 `name` + `description` | 常驻 system prompt | 每个技能几十 token |
| **2** | SKILL.md 正文 | 模型判断任务相关，才整篇读入 | 按需 |
| **3** | 附加文件（`references/`、`scripts/`） | 执行中确需时再读（脚本甚至可以只运行不读入） | 按需 |

官方类比：写 Skill 就像**给新员工准备入职指南**——目录常在手边（Level 1），入职时读总纲（Level 2），用到某项具体流程再翻附件（Level 3）。

先创建技能目录（`%%writefile` 不会自动创建父目录）：


In [16]:
Path("skills/course-summary/references").mkdir(parents=True, exist_ok=True)
Path("skills/flashcard-maker").mkdir(parents=True, exist_ok=True)
print("技能目录就绪：")
for p in sorted(Path("skills").rglob("*")):
    print(f"  {p}")


技能目录就绪：
  skills/course-summary
  skills/course-summary/SKILL.md
  skills/course-summary/references
  skills/course-summary/references/checklist.md
  skills/flashcard-maker
  skills/flashcard-maker/SKILL.md


In [17]:
%%writefile skills/course-summary/SKILL.md
---
name: course-summary
description: 把一节技术课程总结成「学习目标/核心概念/练习/自测题」的标准格式。当用户要求总结课程、复习课程内容时使用。
---

# 课程总结技能

按以下步骤产出课程总结：

1. **学习目标**：3~5 条，每条以动词开头，可观察、可验证。
2. **核心概念**：每个用「一句话定义 + 一个具体例子」解释，保留概念间的区别。
3. **练习**：至少 1 个能用课程现有代码环境直接运行的练习。
4. **自测题**：3 道简答题，先不给答案（用户要求时再给）。

输出用中文 Markdown，加 emoji 小节标题，总长度控制在 500 字以内。
（更详细的验收标准在 references/checklist.md，交稿前再读取。）


Overwriting skills/course-summary/SKILL.md


In [18]:
%%writefile skills/course-summary/references/checklist.md
# 课程总结质量检查清单（Level 3 附件：交稿前才需要读）

- [ ] 学习目标可观察、可验证（避免「了解」「掌握」这类模糊词）
- [ ] 每个核心概念都配了具体例子
- [ ] 练习真的能在课程环境中直接运行
- [ ] 自测题考察了概念间的区别，而不只是背定义
- [ ] 总长度 ≤ 500 字


Overwriting skills/course-summary/references/checklist.md


In [19]:
%%writefile skills/flashcard-maker/SKILL.md
---
name: flashcard-maker
description: 把学习材料做成间隔重复记忆卡片（Anki 风格）。当用户要求做复习卡片、抽认卡、记忆卡时使用。
---

# 记忆卡片技能

每张卡片的格式：

- **正面**：一个问题或一个术语（具体、可直接自测）
- **背面**：不超过 30 字的答案
- 末尾标注该卡片对应的来源小节

一次最多 8 张；优先覆盖「容易混淆的概念对」（如 Tool vs MCP），而不是罗列全部知识点。


Overwriting skills/flashcard-maker/SKILL.md


### 6.1 手写一个 Skill 加载器：亲眼看懂渐进式披露（无需 Key）

三个技能文件已写好（两个 SKILL.md + 一个 Level 3 附件）。现在实现一个最小的加载器——**真实 Agent（如 Claude Code）内部做的就是这三件事**，只是位置在 `.claude/skills/`。


In [20]:
import yaml

def scan_skills(skills_dir: Path) -> dict:
    """Level 1：只加载每个技能的 frontmatter（name + description）——这是常驻上下文的全部。"""
    skills = {}
    for skill_md in sorted(skills_dir.glob("*/SKILL.md")):
        text = skill_md.read_text(encoding="utf-8")
        _, frontmatter, _body = text.split("---", 2)      # 剥出 YAML frontmatter
        meta = yaml.safe_load(frontmatter)
        meta["_dir"] = skill_md.parent
        skills[meta["name"]] = meta
    return skills

SKILLS = scan_skills(Path("skills"))

print("Level 1（常驻 system prompt 的全部内容）：")
resident = ""
for name, meta in SKILLS.items():
    line = f"- {name}: {meta['description']}"
    resident += line
    print("  " + line[:60] + ("..." if len(line) > 60 else ""))

everything = "".join((m["_dir"] / "SKILL.md").read_text(encoding="utf-8") for m in SKILLS.values())
print(f"\n常驻成本 ≈ {len(resident)} 字符；技能包全文 {len(everything)} 字符"
      f"（还不含 Level 3 附件）——差额就是渐进式披露省下的上下文。")

def read_skill(name: str) -> str:
    """Level 2：模型判断任务相关后，才读入正文。"""
    return (SKILLS[name]["_dir"] / "SKILL.md").read_text(encoding="utf-8")

def read_skill_file(name: str, rel_path: str) -> str:
    """Level 3：正文中引用的附件，执行中确需时才读。"""
    return (SKILLS[name]["_dir"] / rel_path).read_text(encoding="utf-8")

print("\nLevel 2 预览（read_skill('course-summary') 正文前 3 行）：")
_body = read_skill("course-summary").split("---", 2)[2]
print("\n".join("  " + l for l in _body.strip().splitlines()[:3]))
print("\nLevel 3 预览（read_skill_file('course-summary', 'references/checklist.md') 前 2 行）：")
print("\n".join("  " + l for l in read_skill_file("course-summary", "references/checklist.md").splitlines()[:2]))


Level 1（常驻 system prompt 的全部内容）：
  - course-summary: 把一节技术课程总结成「学习目标/核心概念/练习/自测题」的标准格式。当用户要求总结课...
  - flashcard-maker: 把学习材料做成间隔重复记忆卡片（Anki 风格）。当用户要求做复习卡片、抽认卡、记...

常驻成本 ≈ 138 字符；技能包全文 590 字符（还不含 Level 3 附件）——差额就是渐进式披露省下的上下文。

Level 2 预览（read_skill('course-summary') 正文前 3 行）：
  # 课程总结技能
  
  按以下步骤产出课程总结：

Level 3 预览（read_skill_file('course-summary', 'references/checklist.md') 前 2 行）：
  # 课程总结质量检查清单（Level 3 附件：交稿前才需要读）
  


### 6.2 让模型自己选技能（需要 Key）

把「读技能」本身做成一个工具，system prompt 里只放 Level 1 元数据，然后观察：**模型会不会只加载与任务相关的那个技能**？这正是 Claude Code 面对几十个技能时的真实机制——技能选择本身也是一个 tool calling 问题。


In [21]:
def read_skill_tool(skill_name: str) -> str:
    """把读技能包装成工具：模型决定何时加载哪本手册。"""
    if skill_name not in SKILLS:
        return f"Error: 未知技能 '{skill_name}'，可用：{list(SKILLS)}"
    return read_skill(skill_name)

SKILL_TOOLS = [{
    "type": "function",
    "function": {
        "name": "read_skill",
        "description": "读取一个技能的完整操作手册（SKILL.md 正文）。执行相关任务前先读手册。",
        "parameters": {
            "type": "object",
            "properties": {
                "skill_name": {"type": "string", "enum": list(SKILLS),
                               "description": "技能名"}
            },
            "required": ["skill_name"],
        },
    },
}]

system_prompt = (
    "你可以调用 read_skill 工具按需加载技能手册。可用技能（仅元数据）：\n"
    + "\n".join(f"- {m['name']}: {m['description']}" for m in SKILLS.values())
    + "\n\n规则：先判断任务匹配哪个技能；匹配则先 read_skill 加载手册再按手册执行；"
      "任务与所有技能都无关时不加载任何技能。"
)

if HAS_KEY:
    agent_loop(
        "请把「Agent 的 Tool、CLI、MCP、Skill 四件套」这节课的内容做成 6 张复习卡片",
        tools=SKILL_TOOLS, registry={"read_skill": read_skill_tool}, system=system_prompt,
    )
    # 观察点：模型应只 read_skill("flashcard-maker")，而不是把两个技能都读进来
else:
    print("⏭️ 未配置 OPENAI_API_KEY，已跳过（6.1 无需 Key 已可运行）。")


[turn 1] 🔧 read_skill({'skill_name': 'flashcard-maker'})
        → ---
name: flashcard-maker
description: 把学习材料做成间隔重复记忆卡片（Anki 风格）。当用户要求做复习卡片、抽认卡、记忆卡时使用。
---

# 记忆卡片技能

每张卡片的格式：

- **正面**
[turn 2] 🏁 最终回答
已按 flashcard-maker 手规范式完成。这四个概念恰好互相易混，因此 6 张卡优先覆盖概念对辨析（Tool vs MCP、Skill vs Tool、CLI vs Tool），而不是平铺罗列：

## 复习卡片 ×6

**卡 1**
- 正面：Tool（工具）是什么？
- 背面：Agent 可直接调用的函数或能力，由模型决定何时使用。
- 来源：Tool 小节

**卡 2**
- 正面：MCP 是什么？
- 背面：统一协议，让 Agent 以标准方式接入外部工具和数据源。
- 来源：MCP 小节

**卡 3**
- 正面：Tool 和 MCP 的区别？
- 背面：Tool 是被调用的具体能力；MCP 是承载和分发能力的标准协议。
- 来源：Tool vs MCP 对比小节

**卡 4**
- 正面：CLI 在 Agent 体系中扮演什么角色？
- 背面：通过命令行与系统交互的接口，Agent 可执行命令完成自动化操作。
- 来源：CLI 小节

**卡 5**
- 正面：Skill 和 Tool 的区别？
- 背面：Skill 是打包的操作说明书（如 SKILL.md），按需加载教模型"怎么做"；Tool 是"能做什么"的执行接口。
- 来源：Skill 小节

**卡 6**
- 正面：为什么 Skill 要按需加载，而不是常驻上下文？
- 背面：节省上下文窗口与 token，只在相关任务出现时才加载。
- 来源：Skill 小节

---

小提示：以上卡片基于这四类概念的标准定义编写。如果你能把这节课的原文或讲义贴给我，我可以核对表述并替换成老师课上的具体说法，保证卡片与课程完全一致。


### 6.3 TongAgents 的 Skill 系统：和开放标准同源

TongAgents 内置了完整的 Skill 系统（`tongagents.tools.skill`）。两个值得注意的设计：

1. **加载的就是 `SKILL.md`**（大小写不敏感）——和 6.1 手写的扫描逻辑、和 Anthropic 的开放标准是同一格式。我们 6.0 写的两个技能文件**原封不动**就能被它加载；
2. **`SkillTool` 把技能清单直接嵌进工具的 description**，模型看到清单后自行决定何时调用 `skill(skill="...", reason="...")` 加载全文——和 6.2 手写的 `read_skill_tool` 是同一个渐进式披露模式，只是连 system_prompt 拼接都不用你写。

另外 `SkillManager.match_skills()` 能按任务描述匹配技能，但它是**按空格分词的关键词匹配**——中文整句（没有空格）匹配不上；生产中真正靠谱的「选技能」还是交给模型（6.4）。

In [22]:
# TongAgents 的 Skill 系统：加载的就是 6.0 写的那两个 SKILL.md
if HAS_TONGAGENTS:
    from tongagents.tools.skill import SkillManager, SkillTool

    # scan_builtin=False / scan_cwd_skills=False：不混入 SDK 内置技能，
    # 只扫描 ./skills —— 教学环境保持干净
    mgr = SkillManager("./skills", scan_builtin=False, scan_cwd_skills=False)
    mgr.load_skills()

    print("=== SkillManager 发现的技能 ===")
    for s in mgr.list_skills():
        print(f"  {s.id}: {s.description[:38]}")

    print("\n=== match_skills 的中文坑：按空格分词 ===")
    print("  整句『帮我总结这门课的要点』→", [s.id for s in mgr.match_skills("帮我总结这门课的要点")] or "（空）")
    print("  空格分词『总结 课程 要点』  →", [s.id for s in mgr.match_skills("总结 课程 要点")])

    st = SkillTool(skills_dir="./skills")
    r = st._do_call({"skill": "course-summary", "reason": "需要总结课程"})
    print("\n=== SkillTool 返回（对照 6.1 的 read_skill：渐进式披露第二级）===")
    print(str(r)[:180].replace("\n", " "), "...")
else:
    print("⏭️ 未安装 tongagents，跳过本节。")


=== SkillManager 发现的技能 ===
  course-summary: 把一节技术课程总结成「学习目标/核心概念/练习/自测题」的标准格式。当用户要
  flashcard-maker: 把学习材料做成间隔重复记忆卡片（Anki 风格）。当用户要求做复习卡片、抽认

=== match_skills 的中文坑：按空格分词 ===
  整句『帮我总结这门课的要点』→ （空）
  空格分词『总结 课程 要点』  → ['course-summary']

=== SkillTool 返回（对照 6.1 的 read_skill：渐进式披露第二级）===
<skill_content name="course-summary"> # course-summary  --- name: course-summary description: 把一节技术课程总结成「学习目标/核心概念/练习/自测题」的标准格式。当用户要求总结课程、复习课程内容时使用。 ---  # 课程总结技能  按以下步骤产出课程总结：  1. ...


### 6.4 让模型自己选技能：ReactAgent + SkillTool

和 6.2 相同的实验，SDK 版只需把 `SkillTool` 当普通工具挂上 `ReactAgent`。观察输出：模型先读到 description 里的技能清单，调用 `skill` 工具加载全文，再按技能模板作答——**整条渐进式披露链路由框架接通**。

In [ ]:
# 和 6.2 相同的实验，SDK 版：把 SkillTool 当普通工具挂上 ReactAgent
if not HAS_TONGAGENTS:
    print("⏭️ 未安装 tongagents，跳过本节。")
elif not HAS_KEY:
    print("⏭️ 未配置 OPENAI_API_KEY，跳过本节。")
else:
    from tongagents.agents.llm.base import ModelConfig
    from tongagents.agents.llm_agent.react_agent import LLMRunContext, ReactAgent, ReactAgentSetting
    from tongagents.tools.skill import SkillTool

    agent = ReactAgent(
        dep_context=LLMRunContext(),
        agent_settings=ReactAgentSetting(llm_config=ModelConfig()),
        tools=[SkillTool(skills_dir="./skills")],
    )
    # 给模型一点真实课程内容，让它按 course-summary 技能的模板输出
    question = (
        "请帮我总结这节课的要点。课程内容：Agent 通过 Tool 以结构化 JSON 表达调用意图，"
        "真正的执行发生在应用代码里；Agent Loop 是「模型出意图→执行→回传→再思考」的循环；"
        "CLI 把命令行变成万能但需要白名单约束的工具；MCP 用开放协议把 M×N 接入问题变成 M+N；"
        "Skill 是按需加载的操作手册，靠渐进式披露节省上下文。"
    )
    result = agent.step(question)
    print("".join(m.content for m in result if getattr(m, "content", None)))


### 6.5 四件套对比：什么时候用哪个

| | Tool | CLI | MCP | Skill |
|---|---|---|---|---|
| **本质** | 单个动作的结构化契约 | 进程级能力接口 | 发现/调用能力的开放协议 | 做事方法的文档包 |
| **解决** | 模型精确传参调用 | 通用、可组合的能力面 | M×N 接入成本 → M+N | 程序性知识注入 |
| **形态** | JSON Schema + 函数 | 命令 + 文本约定 | Server 进程 + JSON-RPC | SKILL.md + 脚本/附件 |
| **上下文成本** | 每个工具的 Schema 常驻 | 一个工具 Schema 常驻 | 发现的 Schema 常驻 | 仅 name+description 常驻 |
| **典型场景** | 业务函数、内部 API | 文件操作、开发工具链 | 跨应用共享能力、第三方服务 | 工作流规范、领域知识 |

它们是**互补**而非互斥：一个成熟 Agent 可以同时挂着工具、一个 shell、几台 MCP server、一摞技能。Anthropic 官方的定位是：Skill 教会 Agent「如何组合外部工具完成复杂工作流」——技能里可以写「先用 MCP 的 github server 查 issue，再用 CLI 工具跑测试」。


## 7. 总结与练习

四句话带走本课：

- **Tool 是一个动作的结构化契约**——模型出意图（JSON），你的代码出结果；
- **CLI 是万能但危险的能力面**——白名单是底线，沙箱与审批是生产标配；
- **MCP 是发现和调用外部能力的开放协议**——M×N 变 M+N，与模型无关；
- **Skill 是按需加载的操作手册**——渐进式披露让上下文花在刀刃上。

而把这四样东西串起来的，始终是同一个东西：**Agent Loop**——模型在循环中基于环境反馈使用工具。


手写白盒与 TongAgents 黑盒的对应关系（本课双线在此收束）：

| 手写实现（各节前半） | TongAgents 对应物 |
|---|---|
| 手写 JSON Schema（2.2） | `@tool` 自动生成（2.5） |
| `TOOL_REGISTRY` + `dispatch_tool`（2.3） | 全局 `ToolManager` 注册表 |
| `agent_loop` while 循环（3.2） | `ReactAgent.step`（3.4） |
| `messages` 手动 append | `SimpleMemory` 自动记录 |
| `run_command` 白名单（4.1） | `ToolHook.check_auth` / `filter_schema` |
| 手写 MCP client + 桥接（5.2–5.3） | `MCPClient` + `MCPToolManager`（5.5） |
| `scan_skills` / `read_skill`（6.1） | `SkillManager`（6.3） |
| `read_skill_tool`（6.2） | `SkillTool`（6.4） |

### 练习（由易到难）

1. 🟢 **加一个工具**：注册 `get_time()`（返回当前时间）到 `TOOL_REGISTRY` 和 `TOOLS`，问模型「现在几点了」，观察它会不会用。
2. 🟢 **强制调用**：把 3.1 节 `chat_once` 的 `tool_choice` 改成 `"required"` 再跑一次，对比行为差异；再试试 `{"type": "function", "function": {"name": "calculate"}}` 强制指定工具。
3. 🟡 **扩展 MCP Server**：给 `mcp_weather_server.py` 加一个 `get_aqi(city)` 工具，重启连接（重跑 5.2 单元格）后确认 `list_tools` 自动发现了它——**客户端一行代码都不用改**，这就是协议的价值。
4. 🟡 **加固 CLI**：给 `run_command` 加上「拒绝包含 `;`、`&&`、`|` 的复合命令」的规则（注意 `|` 管道本来很有用，想想怎么权衡），并测试模型是否还能完成 4.2 的任务。
5. 🔴 **写你自己的 Skill**：写一个 `git-commit-message` 技能（规范：conventional commits + 中文描述），用 6.2 的方法验证模型会正确选中并遵循它。
6. 🔴 **消除硬编码**：把 6.2 的 `system_prompt` 改成完全由 `scan_skills` 的输出自动生成，使得新增技能文件夹后无需改任何代码。
7. 🟢 **SDK 加工具**：用 `@tool` 把 4.1 的 `run_command` 重新实现一遍，交给 `ReactAgent` 完成同样的目录统计任务，对比两版的代码量与安全边界。
8. 🔴 **ToolHook 安全层**：写一个 `ToolHook`——`check_auth` 做命令白名单（对照 4.1 的手写版）、`pre_hook` 自动注入 `timeout` 参数，并验证模型无法调用白名单外的命令。


## 参考资料

**OpenAI 官方**（2026 年起文档主域名是 `developers.openai.com`）

- [Function Calling 指南](https://developers.openai.com/api/docs/guides/function-calling) —— 本课第 2、3 节的 Schema 与消息格式依据
- [Responses vs Chat Completions 迁移指南](https://developers.openai.com/api/docs/guides/responses-vs-chat-completions) —— 两种 API 的字段对照
- [MCP and Connectors 指南](https://developers.openai.com/api/docs/guides/tools-connectors-mcp) —— Responses API 直接挂远程 MCP server
- [模型目录](https://developers.openai.com/api/docs/models) —— 当前模型线与价格
- Cookbook：[How to call functions with chat models](https://developers.openai.com/cookbook/examples/how_to_call_functions_with_chat_models)（天气示例的原出处）
- [Agents SDK 文档](https://openai.github.io/openai-agents-python/) / [MCP 支持](https://openai.github.io/openai-agents-python/mcp/) —— `MCPServerStdio` 等
- 博客：[New tools for building agents](https://openai.com/index/new-tools-for-building-agents/)（Responses API + Agents SDK 发布）、[Apps in ChatGPT & Apps SDK](https://openai.com/index/introducing-apps-in-chatgpt/)（构建于 MCP 之上）
- [Codex CLI](https://github.com/openai/codex) —— OpenAI 的终端编码 Agent

**Anthropic 官方**

- [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) —— workflow vs agent、五种模式、「最简方案优先」哲学
- [Writing tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents) —— 3.3 节最佳实践的出处
- [Equipping agents for the real world with Agent Skills](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills) —— 渐进式披露三级加载
- [Anthropic Tool Use 文档](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

**claude-cookbooks（开源示例仓库）**

- [tool_use/ 系列](https://github.com/anthropics/claude-cookbooks/tree/main/tool_use)：[calculator_tool](https://github.com/anthropics/claude-cookbooks/blob/main/tool_use/calculator_tool.ipynb)（本课 2.1 的原型）、[parallel_tools](https://github.com/anthropics/claude-cookbooks/blob/main/tool_use/parallel_tools.ipynb)、[tool_choice](https://github.com/anthropics/claude-cookbooks/blob/main/tool_use/tool_choice.ipynb)、[customer_service_agent](https://github.com/anthropics/claude-cookbooks/blob/main/tool_use/customer_service_agent.ipynb)
- [skills/ 系列](https://github.com/anthropics/claude-cookbooks/tree/main/skills)：Skills 入门与自定义技能开发

**TongAgents SDK（内部）**

- 源码仓库 `Tong-Agent/`：`tongagents/tools/`（tool_manager / mcp_client / skill）、`tongagents/agents/llm_agent/react_agent.py`
- 文档：`Tong-Agent/docs/sdk/documentation/` 下的 tool.md、agent.md、skill.md（注意文档可能滞后于 2.7.x 代码，以源码为准）
- 安装：BigAI Nexus 私有源 `pip install tongagents tongagents-cli`（配置见 Lesson 1 的 INSTALL.md）

**MCP 与 Skill 标准**

- [Model Context Protocol](https://modelcontextprotocol.io)（[架构](https://modelcontextprotocol.io/docs/concepts/architecture)、[Python SDK](https://github.com/modelcontextprotocol/python-sdk)）
- [Agent Skills 开放标准](https://agentskills.io)


